## Load & Read Data

In [46]:
# import library
import sqlite3
import pandas as pd


#### Helper Method for Database Query

In [47]:
def get_table(table_name):
    conn = sqlite3.connect("ecommerce_hackathon.db")
    try:
        return pd.read_sql_query(
            f'SELECT * FROM "{table_name}"',
            conn
        )
    finally:
        conn.close()


In [48]:
audit = []
def log_issue(table, issue, count, action, reason):
    audit.append({
        "table": table,
        "issue": issue,
        "records_affected": int(count),
        "action": action,
        "reason": reason
    })

#### Print Customers

In [49]:
customers = get_table("customers")
print(customers)

      customer_id  customer_name   age  gender        city signup_date  \
0               1    Hassan Khan  20.0    Male     Karachi  2026-02-27   
1               2     Maham Raza  33.0  Female  Rawalpindi  2025-10-17   
2               3     Ahmed Khan  25.0    Male     Karachi  2022-07-11   
3               4    Salman Khan  26.0    Male      Lahore  2025-02-23   
4               5      Noor Awan  27.0  Female      Lahore  2025-04-21   
...           ...            ...   ...     ...         ...         ...   
7995         7996   Hassan Javed  44.0    Male      Lahore  2022-02-02   
7996         7997    Adeel Saeed  19.0    Male      Lahore  2022-01-24   
7997         7998  Hussain Saeed  31.0    Male    Sargodha  2022-04-30   
7998         7999     Anum Javed  34.0  Female    Sialkot   2025-11-20   
7999         8000    Iqra Sheikh  27.0  Female     Karachi  2022-08-14   

     membership_type                          email             phone  
0           Standard       hassan.khan1

#### Print Products

In [50]:
products = get_table("products")
print(products)

     product_id                    product_name            category  \
0             1     Readings Exam Guide Premium  Books & Stationery   
1             2          Samsung Webcam Classic         Electronics   
2             3  CA Sports Water Bottle Classic    Sports & Fitness   
3             4            LU Spice Mix Classic           Groceries   
4             5            L'Oreal Shampoo 2026              Beauty   
..          ...                             ...                 ...   
995         996          Khaadi Jacket Standard             Fashion   
996         997        L'Oreal Body Lotion 2026              Beauty   
997         998             Shan Detergent Mini           Groceries   
998         999            Rivaj Sunscreen 2026              Beauty   
999        1000                Shan Cereal Mini           Groceries   

         brand  unit_price  stock_quantity  
0     Readings      566.45             103  
1      Samsung    10567.23             138  
2    CA Spor

#### Print Orders

In [51]:
orders = get_table("orders")
print(orders)

       order_id  customer_id  product_id  order_date  quantity  unit_price  \
0             1         2367         389  2026-04-30         1      398.27   
1             2         2038         927  2026-02-09         2    44488.20   
2             3         6285         909  2026-03-30         1     4719.91   
3             4         3714         117  2026-03-24         1     2053.64   
4             5         3021         733  2024-12-13         1    31378.77   
...         ...          ...         ...         ...       ...         ...   
64995     64996         6245         849  2026-08-21         4     3576.73   
64996     64997         3661         407  2026-08-27         1     3891.86   
64997     64998         4387         445  2026-08-28         2     5843.64   
64998     64999         5684         102  2026-06-10         2    35106.59   
64999     65000         3506         944  2024-01-23         1     3230.87   

       discount     payment_method  delivery_days  returned  
0

#### Print Reviews

In [52]:
reviews = get_table("reviews")
print(reviews)

       review_id  customer_id  product_id review_date  rating  \
0              1         7210         787  2026-07-28       4   
1              2         5566         765  2026-05-16       5   
2              3         6106         743  2025-10-21       4   
3              4         2735          90  2026-03-25       4   
4              5         2124         197  2026-02-23       4   
...          ...          ...         ...         ...     ...   
25995      25996         2197         151  2024-05-31       4   
25996      25997         5844         838  2026-08-31       5   
25997      25998           63         664  2024-11-12       4   
25998      25999         4723         278  2026-06-14       3   
25999      26000         6788         151  2026-07-24       4   

                                             review_text  
0      I would buy this again, item feels durable. Re...  
1      Very satisfied, seller description was accurat...  
2      Very satisfied, quality feels premi

# Task A: Database Inspection and Cleaning 

- Showing the shape and columns of each table. 

In [53]:
print(f"\nTable: customers")
print(f"Shape: {customers.shape}")
print(f"Columns: {list(customers.columns)}")
print("-"*140)

print(f"\nTable: products")
print(f"Shape: {products.shape}")
print(f"Columns: {list(products.columns)}")
print("-"*140)

print(f"\nTable: orders")
print(f"Shape: {orders.shape}")
print(f"Columns: {list(orders.columns)}")
print("-"*140)

print(f"\nTable: reviews")
print(f"Shape: {reviews.shape}")
print(f"Columns: {list(reviews.columns)}")
print("-"*140)


Table: customers
Shape: (8000, 9)
Columns: ['customer_id', 'customer_name', 'age', 'gender', 'city', 'signup_date', 'membership_type', 'email', 'phone']
--------------------------------------------------------------------------------------------------------------------------------------------

Table: products
Shape: (1000, 6)
Columns: ['product_id', 'product_name', 'category', 'brand', 'unit_price', 'stock_quantity']
--------------------------------------------------------------------------------------------------------------------------------------------

Table: orders
Shape: (65000, 10)
Columns: ['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'returned']
--------------------------------------------------------------------------------------------------------------------------------------------

Table: reviews
Shape: (26000, 6)
Columns: ['review_id', 'customer_id', 'product_id', 'review_date', 'rating', '

- Customers Cleaning

In [55]:
# Remove accidental whitespace from city names
before_city = customers["city"].copy()

customers["city"] = (
    customers["city"]
    .astype("string")
    .str.strip()
)

changed_city = (
    before_city.astype("string") != customers["city"]
).sum()

log_issue(
    "customers",
    "Leading/trailing whitespace in city",
    changed_city,
    "Trim whitespace",
    "Formatting issue; does not change the underlying value."
)


In [56]:
# Missing age
missing_age = customers["age"].isna().sum()

log_issue(
    "customers",
    "Missing age",
    missing_age,
    "Keep as missing",
    "Age cannot be reliably reconstructed from the available data."
)

In [58]:
# Duplicate emails
duplicate_emails = customers["email"].duplicated(
    keep=False
).sum()

log_issue(
    "customers",
    "Duplicate email addresses",
    duplicate_emails,
    "Keep records",
    "Cannot safely determine which customer record is incorrect."
)

- Products Cleaning

In [61]:
# Standardize category formatting
before_category = products["category"].copy()

products["category"] = (
    products["category"]
    .astype("string")
    .str.strip()
    .str.title()
)

changed_categories = (
    before_category.astype("string") != products["category"]
).sum()

log_issue(
    "products",
    "Inconsistent category capitalization/whitespace",
    changed_categories,
    "Standardize category",
    "These are formatting variations of categorical values."
)


In [62]:
# Missing brand
missing_brand = products["brand"].isna().sum()

log_issue(
    "products",
    "Missing brand",
    missing_brand,
    "Keep as missing",
    "Brand cannot be safely inferred."
)


- Orders Cleaning

In [63]:
# ------------------------------------------------------------
# Order dates
# ------------------------------------------------------------

parsed_dates = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

invalid_order_dates = parsed_dates.isna().sum()

orders["order_date"] = parsed_dates

log_issue(
    "orders",
    "Malformed/unparseable order date",
    invalid_order_dates,
    "Set to missing",
    "Invalid dates cannot be safely reconstructed."
)


# ------------------------------------------------------------
# Quantity
# ------------------------------------------------------------

invalid_quantity = orders["quantity"] <= 0

invalid_quantity_count = invalid_quantity.sum()

# Do NOT delete records.
# Set invalid quantity to missing.
orders.loc[invalid_quantity, "quantity"] = pd.NA

log_issue(
    "orders",
    "Non-positive quantity",
    invalid_quantity_count,
    "Set quantity to missing",
    "A completed sale should have a positive quantity; no safe replacement is available."
)


# ------------------------------------------------------------
# Negative prices
# ------------------------------------------------------------

negative_price = orders["unit_price"] < 0

negative_price_count = negative_price.sum()

# Negative price is treated as a sign error.
# Preserve the record and correct the sign.
orders.loc[negative_price, "unit_price"] = (
    orders.loc[negative_price, "unit_price"].abs()
)

log_issue(
    "orders",
    "Negative unit price",
    negative_price_count,
    "Replace with absolute value",
    "Negative monetary prices are invalid and their magnitudes match normal product prices."
)

# ------------------------------------------------------------
# Missing payment method
# ------------------------------------------------------------

missing_payment = orders["payment_method"].isna().sum()

log_issue(
    "orders",
    "Missing payment method",
    missing_payment,
    "Keep as missing",
    "Payment method cannot be safely inferred."
)


# ------------------------------------------------------------
# Missing delivery days
# ------------------------------------------------------------

missing_delivery = orders["delivery_days"].isna().sum()

log_issue(
    "orders",
    "Missing delivery days",
    missing_delivery,
    "Keep as missing",
    "Delivery duration cannot be reliably reconstructed."
)


In [ ]:
# ------------------------------------------------------------
# Review dates
# ------------------------------------------------------------

parsed_review_dates = pd.to_datetime(
    reviews["review_date"],
    errors="coerce"
)

invalid_review_dates = parsed_review_dates.isna().sum()

reviews["review_date"] = parsed_review_dates

log_issue(
    "reviews",
    "Malformed/unparseable review date",
    invalid_review_dates,
    "Set to missing",
    "Invalid dates cannot be safely reconstructed."
)


# ------------------------------------------------------------
# Ratings
# ------------------------------------------------------------

invalid_rating = ~reviews["rating"].between(1, 5)

invalid_rating_count = invalid_rating.sum()

# Don't convert 0 -> 1 or 6 -> 5.
# That would invent information.
reviews.loc[invalid_rating, "rating"] = pd.NA

log_issue(
    "reviews",
    "Rating outside valid 1-5 range",
    invalid_rating_count,
    "Set rating to missing",
    "Clipping invalid ratings would create fabricated values."
)


# ------------------------------------------------------------
# Review text
# ------------------------------------------------------------

empty_review = (
    reviews["review_text"].isna()
    |
    reviews["review_text"]
    .astype("string")
    .str.strip()
    .eq("")
)

empty_review_count = empty_review.sum()

reviews.loc[empty_review, "review_text"] = pd.NA

log_issue(
    "reviews",
    "Empty or missing review text",
    empty_review_count,
    "Standardize as missing",
    "Empty strings contain no useful text information."
)


# ============================================================
# CHECK REFERENTIAL INTEGRITY
# ============================================================

# Orders -> Customers
orphan_orders_customers = ~orders["customer_id"].isin(
    customers["customer_id"]
)

log_issue(
    "integrity",
    "Orders with nonexistent customer_id",
    orphan_orders_customers.sum(),
    "Verify",
    "Foreign-key relationships should remain intact."
)


# Orders -> Products
orphan_orders_products = ~orders["product_id"].isin(
    products["product_id"]
)

log_issue(
    "integrity",
    "Orders with nonexistent product_id",
    orphan_orders_products.sum(),
    "Verify",
    "Foreign-key relationships should remain intact."
)


# Reviews -> Customers
orphan_reviews_customers = ~reviews["customer_id"].isin(
    customers["customer_id"]
)

log_issue(
    "integrity",
    "Reviews with nonexistent customer_id",
    orphan_reviews_customers.sum(),
    "Verify",
    "Foreign-key relationships should remain intact."
)


# Reviews -> Products
orphan_reviews_products = ~reviews["product_id"].isin(
    products["product_id"]
)

log_issue(
    "integrity",
    "Reviews with nonexistent product_id",
    orphan_reviews_products.sum(),
    "Verify",
    "Foreign-key relationships should remain intact."
)




# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("CLEANING COMPLETE")
print("=" * 60)

print("\nFinal dataset sizes:")
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)
print("Reviews:", reviews.shape)

print("\nFiles created:")
print(OUTPUT_DB)
print(AUDIT_FILE)

print("\nNo customer, order, product, or review records were deleted.")

# Task B: SQL Business Analysis 